# Idea 5 — EN CLIP ViT-B/16 for Attn-last defense

Same 2-model Attn-last intersection protocol as `attention_defense/`, but English
CLIP is **ViT-B/16** (14×14 patches) instead of ViT-B/32 (7×7).

Compare to published B/32 numbers in
`../../attention_defense/results/confusion_results_attn_last.json`.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets', 'matplotlib',
                'Pillow', 'scipy'], check=False)


In [ ]:
import os, platform, random, json, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor

os.makedirs('results', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
BLUR_RADIUS  = 12
THRESHOLDS   = [0.75, 0.80, 0.85, 0.90, 0.95]
PEAKINESS_GATES = [0.05, 0.08, 0.10, 0.12, 0.15]  # max coverage to still skip mask

CLASSES = {
    'en': ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck'],
    'zh': ['飞机','汽车','鸟','猫','鹿','狗','青蛙','马','船','卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}
LANGS = ['en', 'zh']


## 1. Models (EN = ViT-B/16)


In [ ]:
def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    def __init__(self, arch='ViT-B-16'):
        self.arch = arch
        self.m, _, self.pp = open_clip.create_model_and_transforms(arch, pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer(arch)
        # patch grid: B/16 → 14×14, B/32 → 7×7
        self.grid = 14 if '16' in arch else 7
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained(
            'OFA-Sys/chinese-clip-vit-base-patch16',
            attn_implementation='eager').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
        self.grid = 14
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True,
                   return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'],
                                       attention_mask=t['attention_mask'],
                                       token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

def classify_batch(model, imgs, words, batch_size=64):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

print('Loading EN ViT-B-16...', end=' ', flush=True); t0 = time.time()
models = {'en': EnCLIP('ViT-B-16')}
print(f'{time.time()-t0:.1f}s')
print('Loading ZH...', end=' ', flush=True); t0 = time.time()
models['zh'] = ZhCLIP()
print(f'{time.time()-t0:.1f}s')
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in LANGS}
print(f'EN grid={models["en"].grid}×{models["en"].grid}, ZH grid={models["zh"].grid}×{models["zh"].grid}')


## 2. Dataset + attack


In [ ]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000 and np.array_equal(true, np.array(_saved['true']))

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean_224 = [im.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
             for im in rows[image_key]]

all_idx  = np.arange(len(clean_224))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean_224)} images; tune subset = {len(tune_idx)}')

_FONT_CACHE = {}
def _font_paths():
    if platform.system() == 'Windows':
        wf = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        return (os.path.join(wf, 'msyh.ttc'), os.path.join(wf, 'arial.ttf'))
    return None, None
_CJK_FONT, _LAT_FONT = _font_paths()

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:    _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except: _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _rects_overlap(a, b):
    return not (a[2]<=b[0] or b[2]<=a[0] or a[3]<=b[1] or b[3]<=a[1])

def _random_nonoverlapping_rect(rng_, bw, bh, placed):
    x_hi = max(0, DISPLAY_SIZE-bw); y_hi = max(0, DISPLAY_SIZE-bh)
    rx = ry = 0
    for _ in range(64):
        rx = rng_.randint(0, x_hi) if x_hi > 0 else 0
        ry = rng_.randint(0, y_hi) if y_hi > 0 else 0
        rect = (rx, ry, rx+bw, ry+bh)
        if all(not _rects_overlap(rect, p) for p in placed): return rect
    return (rx, ry, rx+bw, ry+bh)

def draw_multilingual_attack(img, en_word, zh_word, img_idx, already_224=False):
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    placed = []
    for box_i, (word, fp) in enumerate([(en_word, _LAT_FONT), (zh_word, _CJK_FONT)]):
        font = _get_font(fp)
        bb   = draw.textbbox((0,0), word, font=font)
        bw   = (bb[2]-bb[0]) + 2*PAD
        bh   = (bb[3]-bb[1]) + PAD + 12
        rng_ = random.Random(int(img_idx)*NUM_BOXES + box_i)
        rect = _random_nonoverlapping_rect(rng_, bw, bh, placed)
        placed.append(rect)
        rx, ry, rx2, ry2 = rect
        draw.rectangle([rx, ry, rx2, ry2], fill='white')
        draw.text((rx+PAD-bb[0], ry+PAD-bb[1]), word, fill='black', font=font)
    return img

attacked_imgs = [
    draw_multilingual_attack(clean_224[i], CLASSES['en'][target[i]],
                              CLASSES['zh'][target[i]], i, already_224=True)
    for i in range(len(clean_224))
]

clean_acc      = {ml: float((classify_batch(models[ml], clean_224, CLASSES[ml]) == true).mean()) for ml in LANGS}
preds_attacked = {ml: classify_batch(models[ml], attacked_imgs, CLASSES[ml]) for ml in LANGS}
baseline_acc   = {ml: float((preds_attacked[ml] == true).mean())   for ml in LANGS}
baseline_asr   = {ml: float((preds_attacked[ml] == target).mean()) for ml in LANGS}
print('Clean acc:   ', {k: f'{100*v:.1f}%' for k,v in clean_acc.items()})
print('Attacked acc:', {k: f'{100*v:.1f}%' for k,v in baseline_acc.items()})
print('Attacked ASR:', {k: f'{100*v:.1f}%' for k,v in baseline_asr.items()})


## 3. Saliency + masking


In [ ]:
def _norm_cam(cam):
    cam = np.maximum(cam if isinstance(cam, np.ndarray) else cam.cpu().numpy(), 0)
    cam -= cam.min(); mx = cam.max()
    return cam / mx if mx > 0 else cam

def align_cam(cam, size=DISPLAY_SIZE):
    return np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(
        (size, size), Image.BILINEAR)) / 255.0

def _make_en_hook(collector):
    def hook(module, inputs, output):
        q_in = inputs[0]
        if getattr(module, 'batch_first', False):
            B, L, D = q_in.shape
        else:
            L, B, D = q_in.shape
            q_in = q_in.transpose(0, 1).contiguous()
        n_head = module.num_heads; hd = D // n_head
        with torch.no_grad():
            qkv = F.linear(q_in, module.in_proj_weight, module.in_proj_bias)
            q, k, _ = qkv.chunk(3, dim=-1)
            q = q.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            k = k.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            attn = ((q @ k.transpose(-2, -1)) * (hd ** -0.5)).softmax(dim=-1)
        collector.append(attn[0].detach().cpu())
    return hook

def _build_attn_cam(all_attns, variant='last'):
    if variant == 'last':
        cls_row = all_attns[-1].mean(0)[0, 1:]
    else:
        L = all_attns[0].shape[-1]
        rollout = torch.eye(L)
        for a in all_attns:
            a_r = 0.5 * a.mean(0) + 0.5 * torch.eye(L)
            a_r = a_r / a_r.sum(-1, keepdim=True)
            rollout = a_r @ rollout
        cls_row = rollout[0, 1:]
    n = int(round(cls_row.shape[0] ** 0.5))
    return _norm_cam(cls_row.reshape(n, n).numpy())

def classify_and_attn_en(pil_img, variant='last'):
    wrapper = models['en']; x = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    collector = []
    handles = [rb.attn.register_forward_hook(_make_en_hook(collector))
               for rb in wrapper.m.visual.transformer.resblocks]
    with torch.no_grad():
        feat = wrapper.m.visual(x)
        imf = F.normalize(feat, dim=-1)
        pred = int((imf @ TEXT_EMB['en'].T).squeeze().argmax().item())
    for h in handles: h.remove()
    return pred, _build_attn_cam(collector, variant)

def classify_and_attn_zh(pil_img, variant='last'):
    wrapper = models['zh']
    pv = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        vis_out = wrapper.m.vision_model(pixel_values=pv, output_attentions=True)
        proj_out = wrapper.m.visual_projection(vis_out.pooler_output)
        imf = F.normalize(proj_out, dim=-1)
        pred = int((imf @ TEXT_EMB['zh'].T).squeeze().argmax().item())
    return pred, _build_attn_cam([a[0].cpu() for a in vis_out.attentions], variant)

def n_cam_intersection(*cams):
    return np.minimum.reduce([align_cam(c) for c in cams])

def dilate_mask(mask, iterations=3):
    m = mask.astype(bool)
    for _ in range(iterations):
        pad = np.pad(m, 1, mode='constant', constant_values=False)
        m = (pad[:-2,:-2]|pad[:-2,1:-1]|pad[:-2,2:]|
             pad[1:-1,:-2]|pad[1:-1,1:-1]|pad[1:-1,2:]|
             pad[2:,:-2]|pad[2:,1:-1]|pad[2:,2:])
    return m

def cam_to_mask(saliency, threshold=0.95, dilate=3):
    thr = np.percentile(saliency, threshold * 100)
    mask = saliency >= thr
    if dilate > 0: mask = dilate_mask(mask, iterations=dilate)
    return mask

def apply_mask(pil_img, mask):
    arr = np.array(pil_img.convert('RGB')); m = mask.astype(bool)
    if mask.shape != arr.shape[:2]:
        m = np.array(Image.fromarray(m.astype(np.uint8)*255).resize(
            arr.shape[1::-1], Image.NEAREST)) > 127
    out = arr.copy()
    mean = arr[~m].mean(0) if (~m).any() else arr.reshape(-1,3).mean(0)
    out[m] = mean
    return Image.fromarray(out.astype(np.uint8))

print('Saliency + mask helpers ready.')


## 4. Tune + full eval


In [ ]:
# Attn-last intersection defense with EN ViT-B/16
THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]

def run_defense(indices, images, threshold, label=''):
    imgs_sub = [images[i] for i in indices]
    true_sub = true[indices]
    print(f'  thr={threshold} {label} n={len(indices)}...', end=' ', flush=True)
    t0 = time.time()
    masks, covs = [], []
    for img in imgs_sub:
        _, cam_en = classify_and_attn_en(img)
        _, cam_zh = classify_and_attn_zh(img)
        inter = n_cam_intersection(cam_en, cam_zh)
        mask  = cam_to_mask(inter, threshold, dilate=3)
        masks.append(mask); covs.append(float(mask.mean()))
    masked = [apply_mask(imgs_sub[j], masks[j]) for j in range(len(imgs_sub))]
    preds = {ml: classify_batch(models[ml], masked, CLASSES[ml]) for ml in LANGS}
    print(f'{time.time()-t0:.1f}s cov={100*np.mean(covs):.1f}%')
    return preds, float(np.mean(covs)), true_sub

# Tune
best_thr, best_acc = 0.95, -1.0
for thr in THRESHOLDS:
    preds, cov, true_sub = run_defense(tune_idx, attacked_imgs, thr, 'tune')
    en = float((preds['en'] == true_sub).mean())
    print(f'    EN={100*en:.1f}%')
    if en > best_acc:
        best_acc, best_thr = en, thr
print(f'Best thr={best_thr} (tune EN={100*best_acc:.1f}%)')

# Full
preds, cov, _ = run_defense(all_idx, attacked_imgs, best_thr, 'full')
cln_preds, _, _ = run_defense(all_idx, clean_224, best_thr, 'clean')

defense = {ml: {
    'acc': float((preds[ml] == true).mean()),
    'asr': float((preds[ml] == target).mean()),
    'baseline_acc': baseline_acc[ml],
    'baseline_asr': baseline_asr[ml],
} for ml in LANGS}
clean_deg = {ml: {
    'baseline_acc': clean_acc[ml],
    'masked_acc': float((cln_preds[ml] == true).mean()),
    'delta_acc': float((cln_preds[ml] == true).mean()) - clean_acc[ml],
} for ml in LANGS}

# Load published B/32 Attn-last for comparison
ref_path = '../../attention_defense/results/confusion_results_attn_last.json'
ref = None
if os.path.exists(ref_path):
    with open(ref_path, encoding='utf-8') as f:
        ref = json.load(f)

row = {
    'setup': 'multilingual',
    'en_arch': 'ViT-B-16',
    'method': 'attn_last_2mod',
    'best_threshold': best_thr,
    'inference_cost': 4,
    'coverage_mean': cov,
    'defense': defense,
    'clean_degradation': clean_deg,
    'defense_acc_mean': float(np.mean([defense[ml]['acc'] for ml in LANGS])),
    'b32_reference': {
        'defense_acc_mean': ref.get('defense_acc_mean') if ref else None,
        'coverage_mean': ref.get('coverage_mean') if ref else None,
        'clean_degradation': ref.get('clean_degradation') if ref else None,
    },
}
with open('results/confusion_results_attn_last_vitb16.json', 'w', encoding='utf-8') as f:
    json.dump(row, f, indent=2, ensure_ascii=False)

print('\nEN ViT-B/16 Attn-last:')
print(f'  mean acc={100*row["defense_acc_mean"]:.1f}%  cov={100*cov:.1f}%')
if ref:
    print(f'  B/32 ref mean={100*ref["defense_acc_mean"]:.1f}%  cov={100*ref["coverage_mean"]:.1f}%')
for ml in LANGS:
    d = clean_deg[ml]
    print(f'  clean {ml}: {100*d["baseline_acc"]:.1f}% → {100*d["masked_acc"]:.1f}% ({100*d["delta_acc"]:+.1f}pp)')

# Bar chart
labels = ['B/32 (published)', 'B/16 (this run)']
means = [
    100 * (ref['defense_acc_mean'] if ref else 0.726),
    100 * row['defense_acc_mean'],
]
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(labels, means, color=['#90a4ae', '#42a5f5'])
ax.set_ylabel('Mean post-defense acc (%)')
ax.set_title('Attn-last: EN ViT-B/32 vs ViT-B/16')
ax.set_ylim(0, 100); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/final_comparison.png', dpi=140)
plt.close()
print('Saved results/final_comparison.png')
